### PMF5 Output Comparison
Summary: This notebook is used to explore methods for comparing the outputs of PMF5 to NMF-PY and development of metrics for evaluating the output of NMF-src.

In [ ]:
import os
import sys
import copy
import logging
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from esat.data.datahandler import DataHandler
from esat.model.base_nmf import BaseSearch
from tests.factor_comparison import FactorComp
from esat.model.optimization import ComponentSearch
from esat.utils import calculate_Q

In [ ]:
n_components = 4
features = 41

pmf_profile_file = os.path.join("D:\\", "projects", "nmf_py", "data", f"baton-rouge_{n_components}f_profiles.txt")
pmf_residuals_file = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test", f"br{n_components}f_residuals.txt")
pmf_contribution_file = os.path.join("D:\\", "projects", "nmf_py", "data", f"baton-rouge_{n_components}f_contributions.txt")

output_path = os.path.join("D:\\", "projects", "nmf_py", "data", "factor_test")
nmf_file = f"nmf-br{n_components}-output.json"
nmf_output_file = os.path.join(output_path, nmf_file)

input_file = os.path.join("D:\\", "projects", "nmf_py", "data", "Dataset-BatonRouge-con.csv")
uncertainty_file = os.path.join("D:\\", "projects", "nmf_py", "data", "Dataset-BatonRouge-unc.csv")

In [ ]:
index_col = "Date"

dh = DataHandler(input_path=input_file,  uncertainty_path=uncertainty_file, output_path=None, index_col=index_col)

In [ ]:
pc = FactorComp(nmf_output_file=nmf_output_file, pmf_profile_file=pmf_profile_file, pmf_contribution_file=pmf_contribution_file, factors=n_components, species=features, residuals_path=pmf_residuals_file)
pmf_q = calculate_Q(pc.pmf_residuals.values, dh.uncertainty_data_processed)
pc.compare(PMF_Q=pmf_q)

In [ ]:
i = 0
nmf_f = pc.factor_map[i]
pmf_f = pc.factor_columns[i]
print(f"PMF factor {pmf_f} is mapped to NMF factor {nmf_f}")

In [ ]:
nmf_H_f = pc.nmf_epochs_dfs[pc.best_model]['H'].loc[nmf_f].to_numpy()    # 41 (features)
nmf_W_f = pc.nmf_epochs_dfs[pc.best_model]['W'][nmf_f].to_numpy()        # 307 (samples)
nmf_W_f = nmf_W_f.reshape(len(nmf_W_f), 1)
nmf_WH_f = np.multiply(nmf_W_f, nmf_H_f)

In [ ]:
pmf_W_f = pc.pmf_contribution_df[pmf_f].to_numpy()
pmf_H_f = pc.pmf_profiles_df[pmf_f].to_numpy()
pmf_W_f = pmf_W_f.reshape(len(pmf_W_f), 1)
pmf_WH_f = np.multiply(pmf_W_f, pmf_H_f)

In [ ]:
corr_matrix = np.corrcoef(nmf_WH_f.flatten(), pmf_WH_f.flatten())
corr = corr_matrix[0, 1]
r_sq = corr ** 2
r_sq

In [ ]:
correlations = []

for i, factor in enumerate(pc.factor_columns):
    nmf_f = pc.factor_map[i]
    pmf_f = factor
    nmf_H_f = pc.nmf_epochs_dfs[pc.best_model]['H'].loc[nmf_f].to_numpy()    # 41 (features)
    nmf_W_f = pc.nmf_epochs_dfs[pc.best_model]['W'][nmf_f].to_numpy()        # 307 (samples)
    nmf_W_f = nmf_W_f.reshape(len(nmf_W_f), 1)
    nmf_WH_f = np.multiply(nmf_W_f, nmf_H_f)
    
    pmf_W_f = pc.pmf_contribution_df[pmf_f].to_numpy()
    pmf_H_f = pc.pmf_profiles_df[pmf_f].to_numpy()
    pmf_W_f = pmf_W_f.reshape(len(pmf_W_f), 1)
    pmf_WH_f = np.multiply(pmf_W_f, pmf_H_f)
    
    corr_matrix = np.corrcoef(nmf_WH_f.flatten(), pmf_WH_f.flatten())
    corr = corr_matrix[0, 1]
    r_sq = corr ** 2
    correlations.append(r_sq)
print(f"Prime Profile - Factor Sample Contributions R2 Avg: {np.mean(correlations)}, Factor R2: {correlations}")

In [ ]:
factor_permutations = list(permutations(pc.factor_columns, len(pc.factor_columns)))
best_factor_mapping = None
best_model = -1
best_avg_r = 0
best_r = []

for model in range(len(pc.nmf_epochs_dfs)):
    for factor_p in factor_permutations:
        r_list = []
        for i, factor in enumerate(pc.factor_columns):
            nmf_factor =  factor_p[i]
            pmf_contribution = pmf_contributions_df[factor]
            nmf_contribution = pc.nmf_epochs_dfs[model]["W"][nmf_factor]
            pmf_contribution = pmf_contribution.astype(float)
            nmf_contribution = nmf_contribution.astype(float)
            corr_matrix = np.corrcoef(nmf_contribution, pmf_contribution)
            corr = corr_matrix[0, 1]
            r2 = corr ** 2
            r_list.append(r2)
        r2_mean = np.mean(r_list)
        if r2_mean > best_avg_r:
            best_avg_r = r2_mean
            best_r = r_list
            best_factor_mapping = factor_p
            best_model = model
print(f"Best Contribution Profile: {best_factor_mapping}, Avg R2: {best_avg_r}, Factor R2: {best_r}")